<a href="https://colab.research.google.com/github/tazmianazar/Decision_Tree_Classifier_Bank_Marketing_Dataset/blob/main/Healthcare_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
HEALTHCARE ADMISSIONS ANALYSIS - END-TO-END PROJECT
====================================================
Steps
  1. Load & first look
  2. Data quality audit
  3. Data cleaning
  4. Feature engineering
  5. Exploratory data analysis (univariate)
  6. Business questions (bivariate / segment analysis)
  7. Statistical hypothesis testing
  8. Predictive modelling (reality check)
  9. Summary of findings

Run:  python healthcare_analysis.py
Outputs: cleaned CSV + PNG charts in ./outputs
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)

DATA_PATH = "/content/healthcare_dataset.csv"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)


def save(fig_name):
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{fig_name}.png", dpi=130)
    plt.close()


# ------------------------------------------------------------------
# STEP 1: LOAD & FIRST LOOK
# ------------------------------------------------------------------
print("\n" + "=" * 70 + "\nSTEP 1: LOAD & FIRST LOOK\n" + "=" * 70)
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print(df.head())
print("\nDtypes:\n", df.dtypes)

# ------------------------------------------------------------------
# STEP 2: DATA QUALITY AUDIT
# ------------------------------------------------------------------
print("\n" + "=" * 70 + "\nSTEP 2: DATA QUALITY AUDIT\n" + "=" * 70)
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("Exact duplicate rows:", df.duplicated().sum())
print("Negative billing rows:", (df["Billing Amount"] < 0).sum())

# Hidden duplicates: rows identical in EVERY column except Age.
# (A plain df.duplicated() misses these because Age differs by 1-5 years.)
_no_age = [c for c in df.columns if c != "Age"]
_exact = df.duplicated().sum()
_near = df.drop_duplicates().duplicated(subset=_no_age).sum()
print(f"Near-duplicate rows (same record, Age altered): {_near}")
print("Identical Billing Amount shared by 2+ rows (a fingerprint of the above):",
      df.duplicated("Billing Amount", keep=False).sum())
print("Names not in proper case:", (df["Name"] != df["Name"].str.title()).sum())
print("Names with titles (Mr., Dr., ...):",
      df["Name"].str.contains(r"^(?:Mr|Mrs|Ms|Dr|Miss)\.?\s", case=False, regex=True).sum())

# ------------------------------------------------------------------
# STEP 3: DATA CLEANING
# ------------------------------------------------------------------
print("\n" + "=" * 70 + "\nSTEP 3: DATA CLEANING\n" + "=" * 70)
n0 = len(df)

# 3.1 Standardise names (they are randomly upper/lower cased)
df["Name"] = df["Name"].str.strip().str.title()

# 3.2 Strip prefixes/suffixes such as "Mr.", "Dr.", "Jr.", "MD" from names
df["Name"] = (df["Name"]
              .str.replace(r"^(?:Mr|Mrs|Ms|Miss|Dr)\.?\s+", "", regex=True)
              .str.replace(r"\s+(?:Jr|Sr|Md|Dds|Phd|Dvm|Ii|Iii|Iv)\.?$", "", regex=True))

# 3.3 Clean hospital text (trailing commas, "and" fragments)
df["Hospital"] = df["Hospital"].str.strip().str.rstrip(",").str.strip()

# 3.4 Parse dates
df["Date of Admission"] = pd.to_datetime(df["Date of Admission"])
df["Discharge Date"] = pd.to_datetime(df["Discharge Date"])

# 3.4b Same title clean-up for doctor names ("Mr. Dean Guzman DDS" -> "Dean Guzman")
df["Doctor"] = (df["Doctor"].str.strip().str.title()
                .str.replace(r"^(?:Mr|Mrs|Ms|Miss|Dr)\.?\s+", "", regex=True)
                .str.replace(r"\s+(?:Jr|Sr|Md|Dds|Phd|Dvm|Ii|Iii|Iv)\.?$", "", regex=True))

# 3.5 Remove duplicates in two passes
#   (a) exact duplicates (after names are standardised so case differences don't hide them)
df = df.drop_duplicates()
n_exact = n0 - len(df)
#   (b) near-duplicates: identical in every column except Age. Keep the first copy.
#       Which Age is "right" is unknowable, so we keep one and note the uncertainty.
df = df.drop_duplicates(subset=[c for c in df.columns if c != "Age"]).reset_index(drop=True)
print(f"Removed {n_exact:,} exact duplicates and {n0 - n_exact - len(df):,} near-duplicates "
      f"-> {len(df):,} rows remain")

# 3.6 Invalid dates (discharge before admission)
bad_dates = (df["Discharge Date"] < df["Date of Admission"]).sum()
print("Rows with discharge before admission:", bad_dates)
df = df[df["Discharge Date"] >= df["Date of Admission"]]

# 3.7 Negative billing amounts = data-entry errors/refunds. Flag, then exclude
#     from financial analysis (keep for clinical analysis).
df["Billing Flag"] = np.where(df["Billing Amount"] < 0, "Negative", "Valid")
print("Negative billing rows flagged:", (df["Billing Flag"] == "Negative").sum())
df["Billing Amount"] = df["Billing Amount"].round(2)

print("Final clean shape:", df.shape)

# ------------------------------------------------------------------
# STEP 4: FEATURE ENGINEERING
# ------------------------------------------------------------------
print("\n" + "=" * 70 + "\nSTEP 4: FEATURE ENGINEERING\n" + "=" * 70)
df["Length of Stay"] = (df["Discharge Date"] - df["Date of Admission"]).dt.days
df["Admit Year"] = df["Date of Admission"].dt.year
df["Admit Month"] = df["Date of Admission"].dt.month
df["Admit Quarter"] = df["Date of Admission"].dt.to_period("Q").astype(str)
df["Admit Weekday"] = df["Date of Admission"].dt.day_name()
df["Age Group"] = pd.cut(df["Age"], bins=[17, 34, 49, 64, 120],
                         labels=["18-34", "35-49", "50-64", "65+"])   # min age is 18 after cleaning
df["Cost per Day"] = df["Billing Amount"] / df["Length of Stay"]

valid = df[df["Billing Flag"] == "Valid"].copy()   # financial analysis subset
df.to_csv(f"{OUT_DIR}/healthcare_clean.csv", index=False)
print(df[["Length of Stay", "Age Group", "Cost per Day"]].head())

# ------------------------------------------------------------------
# STEP 5: UNIVARIATE EDA
# ------------------------------------------------------------------
print("\n" + "=" * 70 + "\nSTEP 5: UNIVARIATE EDA\n" + "=" * 70)
print(df[["Age", "Billing Amount", "Length of Stay"]].describe().round(2))

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(df["Age"], bins=30, kde=True, ax=ax[0]);            ax[0].set_title("Age distribution")
sns.histplot(valid["Billing Amount"], bins=40, kde=True, ax=ax[1]); ax[1].set_title("Billing amount distribution")
sns.histplot(df["Length of Stay"], bins=30, ax=ax[2]);           ax[2].set_title("Length of stay (days)")
save("01_numeric_distributions")

fig, ax = plt.subplots(2, 3, figsize=(16, 8))
for a, col in zip(ax.ravel(), ["Gender", "Blood Type", "Medical Condition",
                               "Admission Type", "Insurance Provider", "Test Results"]):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=a)
    a.set_title(col); a.set_xlabel(""); a.tick_params(axis="x", rotation=30)
save("02_categorical_counts")

# ------------------------------------------------------------------
# STEP 6: BUSINESS QUESTIONS
# ------------------------------------------------------------------
print("\n" + "=" * 70 + "\nSTEP 6: BUSINESS QUESTIONS\n" + "=" * 70)

# Q1. Admission volume trend
monthly = df.set_index("Date of Admission").resample("ME").size()
plt.figure(figsize=(12, 4.5))
monthly.plot()
plt.title("Monthly admissions"); plt.ylabel("Admissions")
save("03_monthly_admissions")
print("Admissions by year:\n", df.groupby("Admit Year").size())
print("(2019 and 2024 are partial years: data runs "
      f"{df['Date of Admission'].min().date()} to {df['Date of Admission'].max().date()})")

# Q2. Which conditions drive cost?
cond = (valid.groupby("Medical Condition")["Billing Amount"]
        .agg(["count", "mean", "median", "sum"]).round(0)
        .sort_values("sum", ascending=False))
print("\nBilling by condition:\n", cond)

plt.figure()
sns.barplot(data=valid, x="Medical Condition", y="Billing Amount", estimator="mean",
            errorbar=("ci", 95), order=cond.sort_values("mean", ascending=False).index)
plt.title("Average billing by condition (95% CI)"); plt.xticks(rotation=20)
save("04_billing_by_condition")

# Q3. Insurance & billing
ins = valid.groupby("Insurance Provider")["Billing Amount"].agg(["count", "mean", "sum"]).round(0)
print("\nBilling by insurer:\n", ins.sort_values("sum", ascending=False))

# Q4. Admission type vs stay & billing
adm = df.groupby("Admission Type").agg(patients=("Name", "size"),
                                       avg_los=("Length of Stay", "mean"),
                                       avg_bill=("Billing Amount", "mean")).round(2)
print("\nBy admission type:\n", adm)

# Q5. Test results by condition
tr = pd.crosstab(df["Medical Condition"], df["Test Results"], normalize="index").round(3) * 100
print("\nTest results % by condition:\n", tr)
plt.figure(figsize=(9, 5))
sns.heatmap(tr, annot=True, fmt=".1f", cmap="Blues")
plt.title("Test result mix (%) by condition")
save("05_test_results_heatmap")

# Q6. Medication by condition (prescribing patterns)
med = pd.crosstab(df["Medical Condition"], df["Medication"], normalize="index").round(3) * 100
print("\nMedication % by condition:\n", med)

# Q7. Age group profile
age_prof = df.groupby("Age Group").agg(patients=("Name", "size"),
                                       avg_bill=("Billing Amount", "mean"),
                                       avg_los=("Length of Stay", "mean")).round(2)
print("\nBy age group:\n", age_prof)

# Q8. Busiest hospitals / doctors (repeat-count check)
print("\nTop 5 hospitals by admissions:\n", df["Hospital"].value_counts().head(5))
print("Distinct hospitals:", df["Hospital"].nunique(), "| distinct doctors:", df["Doctor"].nunique())

# Q9. Correlation
corr = df[["Age", "Billing Amount", "Length of Stay", "Room Number"]].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix")
save("06_correlation")

# Q10. High-cost patient profile (top 10% billing)
thresh = valid["Billing Amount"].quantile(0.90)
valid["High Cost"] = valid["Billing Amount"] >= thresh
print(f"\nTop-10% billing threshold: ${thresh:,.0f}")
for col in ["Medical Condition", "Admission Type", "Insurance Provider", "Age Group"]:
    print(f"\nShare of high-cost patients by {col}:")
    print((valid.groupby(col)["High Cost"].mean() * 100).round(2))

# ------------------------------------------------------------------
# STEP 7: HYPOTHESIS TESTING
# ------------------------------------------------------------------
print("\n" + "=" * 70 + "\nSTEP 7: HYPOTHESIS TESTING (alpha = 0.05)\n" + "=" * 70)


def anova(col, target, data):
    groups = [g[target].values for _, g in data.groupby(col)]
    f, p = stats.f_oneway(*groups)
    return f, p


def chi(a, b):
    chi2, p, dof, _ = stats.chi2_contingency(pd.crosstab(df[a], df[b]))
    return chi2, p


tests = []
for col in ["Medical Condition", "Admission Type", "Insurance Provider"]:
    f, p = anova(col, "Billing Amount", valid)
    tests.append((f"Billing differs by {col}? (ANOVA)", f, p))
f, p = anova("Medical Condition", "Length of Stay", df)
tests.append(("Length of stay differs by condition? (ANOVA)", f, p))
t, p = stats.ttest_ind(valid.loc[valid.Gender == "Male", "Billing Amount"],
                       valid.loc[valid.Gender == "Female", "Billing Amount"])
tests.append(("Billing differs by gender? (t-test)", t, p))
for a, b in [("Medical Condition", "Test Results"), ("Admission Type", "Test Results"),
             ("Medical Condition", "Medication"), ("Gender", "Medical Condition")]:
    c, p = chi(a, b)
    tests.append((f"{a} associated with {b}? (chi-square)", c, p))

res = pd.DataFrame(tests, columns=["Test", "Statistic", "p-value"])
# Running many tests inflates false positives -> Bonferroni-adjust the threshold
res["Significant (raw, p<0.05)"] = np.where(res["p-value"] < 0.05, "YES", "no")
res["Significant (Bonferroni)"] = np.where(res["p-value"] < 0.05 / len(res), "YES", "no")
print(res.round(4).to_string(index=False))
res.to_csv(f"{OUT_DIR}/hypothesis_tests.csv", index=False)

# ------------------------------------------------------------------
# STEP 8: PREDICTIVE MODELLING (reality check)
# ------------------------------------------------------------------
print("\n" + "=" * 70 + "\nSTEP 8: PREDICTIVE MODELLING\n" + "=" * 70)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

feat_cols = ["Age", "Gender", "Blood Type", "Medical Condition", "Admission Type",
             "Insurance Provider", "Medication", "Length of Stay", "Admit Month"]
X = pd.get_dummies(valid[feat_cols], drop_first=True)

# 8a. Predict Test Results (classification) - 5-fold CV vs majority-class baseline
from sklearn.model_selection import StratifiedKFold
y = valid["Test Results"]
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rf = RandomForestClassifier(n_estimators=200, max_depth=10, n_jobs=-1, random_state=42)
acc = cross_val_score(rf, X, y, cv=cv)
base = cross_val_score(DummyClassifier(strategy="most_frequent"), X, y, cv=cv)
print(f"Test Results | RandomForest CV accuracy: {acc.mean():.3f} (+/- {acc.std():.3f}) "
      f"| baseline: {base.mean():.3f}")

# 8b. Predict Billing Amount (regression)
y = valid["Billing Amount"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
rfr = RandomForestRegressor(n_estimators=150, max_depth=10, n_jobs=-1, random_state=42).fit(Xtr, ytr)
pred = rfr.predict(Xte)
base_r = DummyRegressor().fit(Xtr, ytr)
print(f"Billing      | RandomForest R2: {r2_score(yte, pred):.3f}, MAE ${mean_absolute_error(yte, pred):,.0f} | "
      f"baseline MAE ${mean_absolute_error(yte, base_r.predict(Xte)):,.0f}")
print("Interpretation: models at/near baseline => these variables carry no usable predictive signal.")

# ------------------------------------------------------------------
# STEP 9: SUMMARY
# ------------------------------------------------------------------
print("\n" + "=" * 70 + "\nSTEP 9: SUMMARY\n" + "=" * 70)
print(f"""
Records analysed (after cleaning): {len(df):,}
Period: {df['Date of Admission'].min().date()} to {df['Date of Admission'].max().date()}
Total billed (valid rows): ${valid['Billing Amount'].sum():,.0f}
Average bill: ${valid['Billing Amount'].mean():,.0f} | Average stay: {df['Length of Stay'].mean():.1f} days

Charts + cleaned data saved in ./{OUT_DIR}/
""")


STEP 1: LOAD & FIRST LOOK
Shape: (55500, 15)
            Name  Age  Gender Blood Type Medical Condition Date of Admission  \
0  Bobby JacksOn   30    Male         B-            Cancer        2024-01-31   
1   LesLie TErRy   62    Male         A+           Obesity        2019-08-20   
2    DaNnY sMitH   76  Female         A-           Obesity        2022-09-22   
3   andrEw waTtS   28  Female         O+          Diabetes        2020-11-18   
4  adrIENNE bEll   43  Female        AB+            Cancer        2022-09-19   

             Doctor                    Hospital Insurance Provider  \
0     Matthew Smith             Sons and Miller         Blue Cross   
1   Samantha Davies                     Kim Inc           Medicare   
2  Tiffany Mitchell                    Cook PLC              Aetna   
3       Kevin Wells  Hernandez Rogers and Vang,           Medicare   
4    Kathleen Hanna                 White-White              Aetna   

   Billing Amount  Room Number Admission Type Discha